# Analyzing sentence representations in Tucker core projections
We investigate different ways to represent "new" sentences using a precomputed Tucker decomposition of a semantic tensor.

In [1]:
# we reload the packages as we make changes frequently
%load_ext autoreload
%autoreload 2

In [2]:
import itertools
import numpy as np
import tensorflow as tf
import tensorly as tl
import pickle
import torch
import os
from tensorly.decomposition import parafac
from utils import DATA_DIR


def _to_np(x):
    # Accept NumPy arrays or torch tensors; return NumPy view/copy
    if hasattr(x, "detach"):  # torch.Tensor
        return x.detach().cpu().numpy()
    return x

def fetch_latents(factors, vocab, triple):
    # triple = (verb_str, subj_str, obj_str)
    v_idx = vocab["v2i"][triple[0]]
    s_idx = vocab["s2i"][triple[1]]
    o_idx = vocab["o2i"][triple[2]]
    V, S, O = [ _to_np(F) for F in factors ]         # shapes (750,R)
    v = V[v_idx]                                     # (R,)
    s = S[s_idx]                                     # (R,)
    o = O[o_idx]                                     # (R,)
    return v, s, o

def _torch_or_pickle_load(path, map_location="cpu"):
    """Tries to load a torch-saved file, if fails, tries pickle."""
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except RuntimeError as e:
        with open(path, "rb") as f:
            return pickle.load(f)

def cp_decompose_core(core, rank_cp=None, normalize=True):
    """CP-decompose the core: returns (lambdas, U, V, W)."""
    G = _to_np(core)                            # (R,R,R)
    R = G.shape[0]
    if rank_cp is None:
        rank_cp = R
    tl.set_backend('numpy')
    weights, factors = parafac(G, rank=rank_cp, normalize_factors=normalize, init='svd')
    U, V, W = factors                           # each (R, rank_cp); columns are components
    lambdas = _to_np(weights)                   # (rank_cp,)
    return lambdas, _to_np(U), _to_np(V), _to_np(W)

def load_tucker_decomposition(dataset="karrewiet_vectors_ids",
                              method="counting",
                              dims=750,
                              rank=100,
                              iterations=1000,
                              map_location="cpu",  # to load on CPU directly
                              ):
    """Loads a precomputed tucker decomposition from disk.
    Args:
        dataset (str): name of the dataset
        method (str): method used to compute the decomposition
            - one of "counting", "sc", "sii"
        dims (int): dimensionality of the original tensor modes (vocab size)
        rank (int): rank of the decomposition
        iterations (int): number of iterations used to compute the decomposition
       Returns:
        ((core, factors), vocab)
            core: torch.Tensor
            factors: list[torch.Tensor]
            vocab: dict with keys 'vocab_v','vocab_s','vocab_o','v2i','s2i','o2i'
    """
    if method not in {"counting", "sc", "sii"}:
        raise ValueError("method must be one of {'counting','sc','sii'}")

    base = os.path.join(DATA_DIR, "tensors", dataset)
    vocab_path = os.path.join(base, f"vocabularies/{dims}.pkl")
    decomp_path = os.path.join(base, "decomposition", f"{method}_{dims}d_{rank}r_{iterations}i.pt")
    if not os.path.exists(vocab_path):
        raise FileNotFoundError(f"Missing vocab file: {vocab_path}")
    if not os.path.exists(decomp_path):
        raise FileNotFoundError(f"Missing decomposition file: {decomp_path}")
    # the vocab is here under f"vocabularies_[dims].pkl"
    # Load with torch (they were saved with torch.save)

    with open(vocab_path, "rb") as f:
        vocab = pickle.load(f)
    (core, factors) = _torch_or_pickle_load(decomp_path, map_location=map_location)

    return (core, factors), vocab


2025-12-03 14:30:43.746103: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


# One by one explained methods

See obsidian note on sentence representations.


In [3]:
import numpy as np

def score_scalar(core, factors, vocab, triple):
    """(1) Scalar reconstruction score ⟨G, a∘b∘c⟩."""
    G = _to_np(core)                                 # (R,R,R)
    v, s, o = fetch_latents(factors, vocab, triple)
    return np.einsum('pqr,p,q,r->', G, v, s, o)

def contribution_tensor(core, factors, vocab, triple):
    """(2) Contribution tensor: G * (a∘b∘c) ∈ R^{R×R×R}."""
    G = _to_np(core)                                 # (R,R,R)
    v, s, o = fetch_latents(factors, vocab, triple)
    # same as doing np.einsum(p, q, r ->pqr) and then multiplying by G
    return np.einsum('p,q,r,pqr->pqr', v, s, o, G)

def outer_product_latent(factors, vocab, triple):
    """(3) Pseudo-inverse / HOSVD case: a∘b∘c (rank-1 core-space tensor)."""
    v, s, o = fetch_latents(factors, vocab, triple)
    return np.einsum('p,q,r->pqr', v, s, o)



def role_vectors(core, factors, vocab, triple):
    """(4) Role-conditioned 100-d vectors: r_v, r_o, r_s."""
    G = _to_np(core)
    v, s, o = fetch_latents(factors, vocab, triple)
    r_v = np.einsum('pqr,q,r->p', G, s, o)          # leave verb mode
    r_o = np.einsum('pqr,p,r->q', G, v, o)          # leave object mode
    r_s = np.einsum('pqr,p,q->r', G, v, s)          # leave subject mode
    return r_v, r_o, r_s

def bilinear_maps(core, factors, vocab, triple):
    """(5) Bilinear view: Mk = G ×3 c^T ∈ R^{R×R}, score = a^T Mk b.
       Also return 100-d vectors u_v = Mk b and v_o = Mk^T a."""
    G = _to_np(core)
    v, s, o = fetch_latents(factors, vocab, triple)
    Mk = np.einsum('pqr,r->pq', G, o)               # (R,R)
    score = float(v @ (Mk @ s))
    u_v = Mk @ s                                    # (R,)
    v_o = Mk.T @ v                                  # (R,)
    return score, Mk, u_v, v_o





def component_contributions_via_cp_core(core, factors, vocab, triple, rank_cp=None, return_diagnostics=True):
    """
    Component-wise contribution vector for a triple using a CP of the core.

    Inputs
    ------
    core:         (R,R,R) core tensor (torch or np)
    factors:      [V, S, O] with shapes (Nv,R), (Ns,R), (No,R)
    vocab:        dict with 'v2i','s2i','o2i'
    triple:       (verb_str, subj_str, obj_str)
    cp_cache:     tuple (lambdas, U, V, W) for the core, from cp_decompose_core; optional
    rank_cp:      CP rank to use if cp_cache is None. If None, defaults to R.
    return_diagnostics: if True, also return score approximation and exact score.

    Returns
    -------
    t:            (R_cp,) vector with t[r] = λ_r (a·U[:,r]) (b·V[:,r]) (c·W[:,r])
    diag (opt):   dict with 'score_cp', 'score_exact', 'score_diff'
    """
    # Fetch triple latents a,b,c
    v, s, o = fetch_latents(factors, vocab, triple)   # each (R,)

    # Get or build CP of the core
    # if cp_cache is None:
    #     lambdas, U, V, W = cp_decompose_core(core, rank_cp=rank_cp)
    # else:
    #     lambdas, U, V, W = cp_cache
    lambdas, U, V, W = cp_decompose_core(core, rank_cp=rank_cp)

    # Projections onto CP components
    alpha = U.T @ v     # (R_cp,)
    beta  = V.T @ s     # (R_cp,)
    gamma = W.T @ o     # (R_cp,)

    # Per-component contributions
    t = lambdas * alpha * beta * gamma                 # (R_cp,)

    if not return_diagnostics:
        return t

    # Optional diagnostics: CP-approximated score vs exact scalar reconstruction
    score_cp    = float(np.sum(t))
    score_exact = float(np.einsum('pqr,p,q,r->', _to_np(core), v, s, o))
    diag = {"score_cp": score_cp,
            "score_exact": score_exact,
            "score_diff": score_cp - score_exact}
    return t, diag


def kronecker_projected(core, factors, vocab, triple, W=None):
    """(7) Kronecker feature with projection.
       If W is None -> returns scalar score (same as (1)).
       If W has shape (d,R,R,R), returns h ∈ R^d with h[d] = ⟨W[d], a∘b∘c⟩.
    """
    v, s, o = fetch_latents(factors, vocab, triple)
    if W is None:
        # same as score_scalar but without touching core
        return np.einsum('p,q,r,pqr->', v, s, o, _to_np(core))
    Wnp = _to_np(W)           # (d,R,R,R)
    h = np.einsum('dpqr,p,q,r->d', Wnp, v, s, o)
    return h

def balanced_vector(core, factors, vocab, triple, leave='subject'):
    """(8) Leave-one-mode-uncontracted vector (100-d).
       leave ∈ {'verb','object','subject'}.
    """
    G = _to_np(core)
    v, s, o = fetch_latents(factors, vocab, triple)
    if leave == 'verb':
        return np.einsum('pqr,q,r->p', G, s, o)     # same as role_vectors()[0]
    if leave == 'object':
        return np.einsum('pqr,p,r->q', G, v, o)
    if leave == 'subject':
        return np.einsum('pqr,p,q->r', G, v, s)
    raise ValueError("leave must be one of {'verb','object','subject'}")


def slice_pooling_features(core, factors, vocab, triple, variant='Si_c'):
    """(9) Slice-pooling features.
       Variants:
         - 'score' : scalar a^T (G×1 a^T) b with c; i.e., full score
         - 'Si_c'  : vector (G×1 a^T) @ c           ∈ R^R
         - 'Si_Tb' : vector (G×1 a^T)^T @ b         ∈ R^R
         - 'diag'  : vector diag(G×1 a^T)           ∈ R^R
    """
    G = _to_np(core)
    v, s, o = fetch_latents(factors, vocab, triple)
    S_i = np.einsum('pqr,p->qr', G, v)              # (R,R)
    if variant == 'score':
        return float(s @ (S_i @ o))                 # same scalar as (1)
    if variant == 'Si_c':
        return S_i @ o                              # (R,)
    if variant == 'Si_Tb':
        return S_i.T @ s                            # (R,)
    if variant == 'diag':
        return np.diag(S_i)                         # (R,)
    raise ValueError("variant must be one of {'score','Si_c','Si_Tb','diag'}")


def probabilistic_wrappers(core, factors, vocab, triple, link='logistic'):
    """(10) Probabilistic wrappers for the scalar score.
       link ∈ {'logistic','softplus','identity'}:
         - logistic: σ(score)
         - softplus: log(1+exp(score)) as a nonneg. rate
         - identity: raw score
    """
    s = score_scalar(core, factors, vocab, triple)
    if link == 'logistic':
        return 1.0 / (1.0 + np.exp(-s))
    if link == 'softplus':
        # numerically stable softplus
        if s > 0:
            return s + np.log1p(np.exp(-s))
        else:
            return np.log1p(np.exp(s))
    if link == 'identity':
        return s
    raise ValueError("link must be one of {'logistic','softplus','identity'}")


In [4]:
# Testing this out
(core, factors), vocab = load_tucker_decomposition(dataset="fineweb_sparse",
                              method="sc",
                              dims=2500,
                              rank=100,
                              iterations=10000,
                              map_location="cpu", )
print(core.shape, [F.shape for F in factors])
print("Vocab sizes:", len(vocab['v2i']), len(vocab['s2i']), len(vocab['o2i']))
test_sentences = [
    ("eten", "man", "ontbijt"),
    ("lezen", "vrouw", "boek"),
    ("rijden", "kind", "fiets"),
    ("lezen", "hond", "bed"),
    ("bombarderen", "leger", "stad"),
    ("bouwen", "man", "gebouw"),
    ("bouwen", "kat", "boek")
    ]
# we check if all words are in the relevant vocabularies
for verb, subj, obj in test_sentences:
    assert verb in vocab['v2i'], f"Verb '{verb}' not in vocab"
    assert subj in vocab['s2i'], f"Subject '{subj}' not in vocab"
    assert obj in vocab['o2i'], f"Object '{obj}' not in vocab"

torch.Size([100, 100, 100]) [torch.Size([2500, 100]), torch.Size([2500, 100]), torch.Size([2500, 100])]
Vocab sizes: 2500 2500 2500


In [5]:
G = _to_np(core)
v, s, o = fetch_latents(factors, vocab, test_sentences[0])
a = np.einsum("p,q,r,pqr->pqr", v, s, o, G)
b = np.einsum("p,q,r->pqr", v, s, o)
c = b*G
# we print if all are equal
print("Are weighted and unweighted equal?", np.allclose(a, b))
print("Are weighted and core-multiplied equal?", np.allclose(a, c))

# we sum over all entries in a
print("Sum of all entries in weighted:", np.sum(a))
d = score_scalar(core, factors, vocab, test_sentences[0])
print("Scalar score:", d)

Are weighted and unweighted equal? False
Are weighted and core-multiplied equal? True
Sum of all entries in weighted: 0.012608665
Scalar score: 0.012608666


In [6]:
tup = ("spelen", "jongen", "spel")
a = role_vectors(core, factors, vocab, tup)
print("Role vectors shapes:", [vec.shape for vec in a])
verb_vec = a[0]
print("Verb role vector (first 10 dims):", verb_vec[:10])
# we compare this to the latent factor representation of the verb
v_latent, _, _ = fetch_latents(factors, vocab, tup)
print("Verb latent vector (first 10 dims):", v_latent[:10])
# we compute the cosine similarity between the two
cos_sim = np.dot(verb_vec, v_latent) / (np.linalg.norm(verb_vec) * np.linalg.norm(v_latent))
print("Cosine similarity between verb role vector and latent vector:", cos_sim)


Role vectors shapes: [(100,), (100,), (100,)]
Verb role vector (first 10 dims): [5.8090184e-03 4.2088708e-05 4.8608560e-04 2.0293464e-04 5.5406534e-04
 1.6484848e-03 2.7506423e-04 6.1091525e-04 2.8843537e-05 1.2538406e-04]
Verb latent vector (first 10 dims): [0.0000000e+00 0.0000000e+00 0.0000000e+00 0.0000000e+00 0.0000000e+00
 0.0000000e+00 0.0000000e+00 0.0000000e+00 0.0000000e+00 4.1578005e-14]
Cosine similarity between verb role vector and latent vector: 0.1131766


In [7]:

G = _to_np(core)
for sentence in test_sentences:
    print(f"\n{sentence}")
    v, s, o = fetch_latents(factors, vocab, sentence)
    weighted_vector = np.einsum("pqr,p,q,r->p", G, v, s, o)
    unweighted_vector = np.einsum("p,q,r->p", v, s, o)
    print("Weighted vector (first 10 dims):", weighted_vector[:10])
    print("Unweighted vector (first 10 dims):", unweighted_vector[:10])
    # we compute the cosine similarity between the two
    cos_sim_weighted = np.dot(weighted_vector, unweighted_vector) / (np.linalg.norm(weighted_vector) * np.linalg.norm(unweighted_vector))
    print("Cosine similarity between weighted and unweighted vectors:", cos_sim_weighted)



('eten', 'man', 'ontbijt')
Weighted vector (first 10 dims): [3.8400175e-11 1.0040635e-05 1.9935617e-06 0.0000000e+00 2.0785152e-04
 0.0000000e+00 0.0000000e+00 2.3925060e-03 3.6000449e-09 3.7990276e-07]
Unweighted vector (first 10 dims): [1.2160098e-08 4.0209759e-02 1.6149150e-02 0.0000000e+00 2.5589031e-01
 0.0000000e+00 0.0000000e+00 3.4789730e-02 2.2794101e-03 1.1905522e-02]
Cosine similarity between weighted and unweighted vectors: 0.4190332

('lezen', 'vrouw', 'boek')
Weighted vector (first 10 dims): [0.0000000e+00 0.0000000e+00 3.2064472e-05 1.3119683e-05 6.4294880e-05
 0.0000000e+00 0.0000000e+00 0.0000000e+00 0.0000000e+00 4.6959838e-07]
Unweighted vector (first 10 dims): [0.         0.         0.3729236  0.7124673  0.1804429  0.
 0.         0.         0.         0.14411253]
Cosine similarity between weighted and unweighted vectors: 0.85345113

('rijden', 'kind', 'fiets')
Weighted vector (first 10 dims): [5.4973236e-05 1.1816620e-06 0.0000000e+00 0.0000000e+00 0.0000000e+00
 0

# Re-integrating one of the factors into the core
If we re-integrate one of the factors back into the core, we can "read", for example, the expected subject and object values for a given verb.
Or do we? this is the experiment

In [8]:
print(core.shape, factors[0].shape, factors[1].shape, factors[2].shape)

torch.Size([100, 100, 100]) torch.Size([2500, 100]) torch.Size([2500, 100]) torch.Size([2500, 100])


In [9]:
# we reintegrate the V factor (verbs) into the core
G = _to_np(core)
# multiply the core by the verb factor matrix A along mode-1,
G_verb = np.einsum('ip, pqr -> iqr', _to_np(factors[0]), G)   # (Nv,R,R)
print("New core shape after reintegrating verbs:", G_verb.shape)

New core shape after reintegrating verbs: (2500, 100, 100)


In [10]:
# we check the values for the verb "eten"
verb = "eten"
verb_idx = vocab["v2i"][verb]
G_eten = G_verb[verb_idx, :, :]    # (R,R)
# we can check how similar this is to the "actual" seen values for subjects and objects with this verb in a given sentence
test_sentences_eten = [
    ("eten", "man", "ontbijt"),
    ("eten", "vrouw", "kip"),
    ("schieten", "hij", "doel"),
    ("schieten", "koe", "boerderij")
    ]
for sentence in test_sentences_eten:
    print(f"\nSentence: {sentence}")
    v, s, o = fetch_latents(factors, vocab, sentence)
    # we compute the similarity between the G_eten and the outer product of s and o
    outer_so = np.einsum('q,r->qr', s, o)
    weighted_outer_so = np.einsum('p,q,r,pqr->qr', v, s, o, G)
    masked_G_eten = G_eten * outer_so

    # we check if masked_G_eten is equal to any of the other
    print("Is masked_G_eten equal to weighted_outer_so?", np.allclose(masked_G_eten, weighted_outer_so))
    # we calculate cosine distance between masked_G_eten and weighted_outer_so
    # and between outer_so and the weighted_outer_so
    cos_sim = np.sum(masked_G_eten * weighted_outer_so) / (np.linalg.norm(masked_G_eten) * np.linalg.norm(weighted_outer_so))
    print("Cosine similarity between masked_G_eten and weighted_outer_so:", cos_sim)
    cos_sim_outer = np.sum(outer_so * weighted_outer_so) / (np.linalg.norm(outer_so) * np.linalg.norm(weighted_outer_so))
    print("Cosine similarity between outer_so and weighted_outer_so:", cos_sim_outer)



Sentence: ('eten', 'man', 'ontbijt')
Is masked_G_eten equal to weighted_outer_so? True
Cosine similarity between masked_G_eten and weighted_outer_so: 1.0000001
Cosine similarity between outer_so and weighted_outer_so: 0.27980223

Sentence: ('eten', 'vrouw', 'kip')
Is masked_G_eten equal to weighted_outer_so? True
Cosine similarity between masked_G_eten and weighted_outer_so: 1.0
Cosine similarity between outer_so and weighted_outer_so: 0.22971164

Sentence: ('schieten', 'hij', 'doel')
Is masked_G_eten equal to weighted_outer_so? False
Cosine similarity between masked_G_eten and weighted_outer_so: 0.6951452
Cosine similarity between outer_so and weighted_outer_so: 0.5366926

Sentence: ('schieten', 'koe', 'boerderij')


KeyError: 'koe'

## Old code
We built up simple sentence representations by projecting the latent factors onto the core tensor.
The underlying idea is to "reconstruct" the original tensor slice corresponding to the given triple
based on the most important information ~latent factors of the learned decomposition.

In [20]:
# we use the cpu core and factors
def create_core_projection(core, factors, vocab, test_tuple, weighted=True):
    verb_idx = vocab["v2i"][test_tuple[0]]
    subj_idx = vocab["s2i"][test_tuple[1]]
    obj_idx = vocab["o2i"][test_tuple[2]]
    v = factors[0].cpu()[verb_idx, :]
    s = factors[1].cpu()[subj_idx, :]
    o = factors[2].cpu()[obj_idx, :]
    if weighted:
        # elementwise multiply the core by the outer-product weights
        # shape (R1,R2,R3): v[i]*s[j]*o[k] * core[i,j,k]
        proj = np.einsum('i,j,k,ijk->ijk', v, s, o, core.cpu().numpy())
    else:
        proj = np.einsum('i,j,k->ijk', v, s, o)
    return proj


In [21]:
proj = create_core_projection(core, factors, vocab, ("eten", "man", "ontbijt"))
# we print if there are any non-zero values
print("Non-zero values in projection:", np.count_nonzero(proj))
# we print their indices and values
non_zero_indices = np.argwhere(proj != 0)
# for each mode, we count how many times each index appears in the non-zero indices
mode0_counts = {}
mode1_counts = {}
mode2_counts = {}
for idx in non_zero_indices:
    i, j, k = idx
    mode0_counts[i] = mode0_counts.get(i, 0) + 1
    mode1_counts[j] = mode1_counts.get(j, 0) + 1
    mode2_counts[k] = mode2_counts.get(k, 0) + 1
print("Mode 0 (verbs) non-zero index counts:", mode0_counts)
print("Mode 1 (subjects) non-zero index counts:", mode1_counts)
print("Mode 2 (objects) non-zero index counts:", mode2_counts)

Non-zero values in projection: 118660
Mode 0 (verbs) non-zero index counts: {0: 1220, 1: 1221, 2: 1080, 3: 1217, 4: 1211, 5: 1220, 6: 1209, 7: 1245, 8: 1184, 9: 1204, 10: 1229, 11: 1100, 12: 1228, 13: 1152, 14: 1176, 15: 1110, 16: 1151, 17: 1219, 18: 1237, 19: 1223, 20: 1217, 21: 1217, 22: 1220, 23: 1161, 24: 1225, 25: 1144, 26: 1203, 27: 1216, 28: 1226, 29: 1211, 30: 1200, 31: 1203, 32: 1164, 33: 1213, 34: 1195, 35: 1233, 36: 1211, 37: 1225, 38: 1162, 39: 1130, 40: 1216, 41: 1227, 42: 1169, 43: 1213, 44: 1073, 45: 1139, 46: 1234, 47: 1221, 48: 1222, 49: 1222, 50: 1226, 51: 1233, 52: 1188, 53: 1086, 54: 1084, 55: 1235, 56: 1213, 57: 1192, 58: 1226, 59: 1160, 60: 1231, 61: 1232, 62: 1230, 63: 1064, 64: 1181, 65: 1108, 66: 1185, 67: 1210, 68: 1234, 69: 1248, 70: 1089, 71: 1109, 72: 1154, 73: 1205, 74: 1238, 75: 1235, 76: 696, 77: 1090, 78: 1222, 79: 1197, 80: 1059, 81: 1077, 82: 1214, 83: 1251, 84: 1236, 85: 1223, 86: 1226, 87: 1148, 88: 1226, 89: 1220, 90: 1214, 91: 1232, 92: 1205, 93: 

In [22]:
proj_unweighted = create_core_projection(core, factors, vocab, ("eten", "man", "ontbijt"), weighted=False)
# we print if there are any non-zero values
print("Non-zero values in projection:", np.count_nonzero(proj))
# we print their indices and values
non_zero_indices_unweighted = np.argwhere(proj_unweighted != 0)
# for each mode, we count how many times each index appears in the non-zero indices
mode0_counts_unweighted = {}
mode1_counts_unweighted = {}
mode2_counts_unweighted = {}
for idx in non_zero_indices_unweighted:
    i, j, k = idx
    mode0_counts_unweighted[i] = mode0_counts.get(i, 0) + 1
    mode1_counts_unweighted[j] = mode1_counts.get(j, 0) + 1
    mode2_counts_unweighted[k] = mode2_counts.get(k, 0) + 1
print("Mode 0 (verbs) non-zero index counts:", mode0_counts_unweighted)
print("Mode 1 (subjects) non-zero index counts:", mode1_counts_unweighted)
print("Mode 2 (objects) non-zero index counts:", mode2_counts_unweighted)

Non-zero values in projection: 118660
Mode 0 (verbs) non-zero index counts: {0: 1221, 1: 1222, 2: 1081, 3: 1218, 4: 1212, 5: 1221, 6: 1210, 7: 1246, 8: 1185, 9: 1205, 10: 1230, 11: 1101, 12: 1229, 13: 1153, 14: 1177, 15: 1111, 16: 1152, 17: 1220, 18: 1238, 19: 1224, 20: 1218, 21: 1218, 22: 1221, 23: 1162, 24: 1226, 25: 1145, 26: 1204, 27: 1217, 28: 1227, 29: 1212, 30: 1201, 31: 1204, 32: 1165, 33: 1214, 34: 1196, 35: 1234, 36: 1212, 37: 1226, 38: 1163, 39: 1131, 40: 1217, 41: 1228, 42: 1170, 43: 1214, 44: 1074, 45: 1140, 46: 1235, 47: 1222, 48: 1223, 49: 1223, 50: 1227, 51: 1234, 52: 1189, 53: 1087, 54: 1085, 55: 1236, 56: 1214, 57: 1193, 58: 1227, 59: 1161, 60: 1232, 61: 1233, 62: 1231, 63: 1065, 64: 1182, 65: 1109, 66: 1186, 67: 1211, 68: 1235, 69: 1249, 70: 1090, 71: 1110, 72: 1155, 73: 1206, 74: 1239, 75: 1236, 76: 697, 77: 1091, 78: 1223, 79: 1198, 80: 1060, 81: 1078, 82: 1215, 83: 1252, 84: 1237, 85: 1224, 86: 1227, 87: 1149, 88: 1227, 89: 1221, 90: 1215, 91: 1233, 92: 1206, 93: 